## Question 5 : Algorithme retenu

L'objectif est de minimiser le temps de séjour moyen. Ainsi, il est nécessaire de s'intéresser au temps écoulé entre l'arrivée d'un produit dans l'atelier et sa sortie de ce-dernier. 

Premièrement, nous avons opté pour l'algortihme suivant. L'idée est de considérer un employé quelconque avec des qualifications qui lui sont propres. On parcourt, ensuite, les machines pour lesquelles il est qualifié et dont la file d'attente est non vide. Enfin, on affecte l'employé à la machine dont le produit en tête de file a le temps de traitement estimé le plus court.

Cet algorithme se justifie par l'intuition suivante :
Considérons un système de files et un serveur. Faire en sorte de traiter d'abord les tâches courtes assure de libérer rapidement des produits, ce qui réduit la longueur moyenne des files et donc le temps d'attente de tous les produits suivants.
En revanche, un problème peut se poser dans l'atelier du fait de sa complexité (réseau de files, qualifications hétérogènes, ...). En effet, supposons qu'un produit attend depuis assez longtemps dans une file dû à son grand temps de traitement. Si on décide d'appliquer seulement l'intuition précédente, ce produit ne sera jamais prioritaire et il se peut que les employés traitent indéfiniment des tâches plus courtes sans ne jamais s'attarder sur ce produit. Ainsi, la file dont est issue ce produit n'avancera pas. Cela pose problème. Ainsi, l'idée est de comparer l'âge des produits en tête de file à leur temps de traitement estimé. On obtient un score qui oriente l'employé. Effectivement, on affecte celui-ci à la machine qui maximise le score.
Cela fonctionne en définissant le score comme le rapport entre l'âge du produit et sa durée de traitement estimée. Plus l'âge du produit est grand, plus le produit est urgent. A urgence égale, l'employé préfère la tâche qu'il peut terminer le plus vite.

## Question 6 : Implémentation et résultats

L'algorithme retenu, codé en Julia, est comme suit :

In [ ]:
using SimJulia, Distributions, Random, Statistics, ResumableFunctions, Printf
using Plots

# --- DONNÉES OFFICIELLES (inchangées) ---
const λ = [0.29, 0.32, 0.47, 0.38]
const PARCOURS = [
    [1, 2, 3, 4, 8], # T1
    [2, 4, 7],       # T2
    [3, 5, 1],       # T3
    [5, 6, 7, 8]     # T4
]
const TEMPS = Dict(
    (1,1)=>(0.58,0.78), (1,2)=>(0.23,0.56), (1,3)=>(0.81,0.93), (1,4)=>(0.12,0.39), (1,8)=>(0.82,1.04),
    (2,2)=>(0.59,0.68), (2,4)=>(0.74,0.77), (2,7)=>(0.30,0.55),
    (3,3)=>(0.37,0.54), (3,5)=>(0.35,0.63), (3,1)=>(0.57,0.64),
    (4,5)=>(0.36,0.51), (4,6)=>(0.61,0.70), (4,7)=>(0.78,0.85), (4,8)=>(0.18,0.37)
)

# --- STRUCTURES ---
mutable struct Produit
    id::Int
    type::Int
    arrivee_atelier::Float64
    arrivee_machine::Float64
    etape::Int
end

mutable struct Machine
    file_attente::Vector{Produit}
    en_service::Union{Produit,Nothing}
    occupee::Bool
    Machine() = new(Produit[], nothing, false)
end

mutable struct Employe
    id::Int
    qualifications::Vector{Int}
    temps_entree_salle::Float64
    est_occupe::Bool
    travail_cumule::Float64
    machine_assignee::Union{Int,Nothing}
    Employe(id, quals) = new(id, quals, 0.0, false, 0.0, nothing)
end

mutable struct Atelier
    sim::Simulation
    machines::Vector{Machine}
    employes::Vector{Employe}
    file_inactifs::Vector{Int}
    events_employes::Vector{Event}
    temps_sejour::Vector{Float64}
    temps_sejour_history::Vector{Tuple{Float64,Float64}}
    compteur_produits::Int
    fin_transitoire::Float64
    verbose::Bool
    record_history::Bool
    function Atelier(sim, Q, verbose, record_history)
        nb_machines = 8
        nb_emp = size(Q,1)
        machines = [Machine() for _ in 1:nb_machines]
        employes = [Employe(i, findall(j -> Q[i,j]==1, 1:nb_machines)) for i in 1:nb_emp]
        events = [Event(sim) for _ in 1:nb_emp]
        new(sim, machines, employes, Int[], events, Float64[], Tuple{Float64,Float64}[], 0, 0.0, verbose, record_history)
    end
end

# --- FONCTION DE LOG ---
log_event(atelier::Atelier, msg::String) = atelier.verbose && println("[t=$(Printf.@sprintf("%.3f", now(atelier.sim)))] $msg")

# --- FONCTIONS AUXILIAIRES ---

function par_temps_entree(atelier::Atelier, id::Int)
    return atelier.employes[id].temps_entree_salle
end

function index_produit_doyen(file::Vector{Produit})
    isempty(file) && return nothing
    idx_min = 1
    val_min = file[1].arrivee_atelier
    for i in 2:length(file)
        val = file[i].arrivee_atelier
        if val < val_min
            val_min = val
            idx_min = i
        end
    end
    return idx_min
end

function entrer_salle_attente!(atelier::Atelier, emp_id::Int, temps::Float64)
    emp = atelier.employes[emp_id]
    emp.est_occupe = false
    emp.temps_entree_salle = temps
    push!(atelier.file_inactifs, emp_id)
    sort!(atelier.file_inactifs, by = id -> par_temps_entree(atelier, id))
    log_event(atelier, "Employé $emp_id entre en salle d'attente (file=$(atelier.file_inactifs))")
    return nothing
end

function sortir_salle_attente!(atelier::Atelier, emp_id::Int)
    filter!(e -> e != emp_id, atelier.file_inactifs)
    atelier.employes[emp_id].est_occupe = true
    log_event(atelier, "Employé $emp_id sort de la salle d'attente")
end

function choisir_employe_pour_machine(atelier::Atelier, machine_id::Int)
    for emp_id in atelier.file_inactifs
        emp = atelier.employes[emp_id]
        if machine_id in emp.qualifications
            sortir_salle_attente!(atelier, emp_id)
            return emp_id
        end
    end
    return nothing
end

function choisir_machine(atelier::Atelier, emp_id::Int,
                         mode_algo::String,
                         mode_choix::String)

    emp_ref = atelier.employes[emp_id]

    # ---- MODE FIFO ----
    if mode_algo == "FIFO_ATELIER"
        meilleure_machine = nothing
        arrivee_min = Inf
        for m_id in emp_ref.qualifications
            machine = atelier.machines[m_id]
            isempty(machine.file_attente) && continue
            idx = index_produit_doyen(machine.file_attente)
            t = machine.file_attente[idx].arrivee_atelier
            if t < arrivee_min
                arrivee_min = t
                meilleure_machine = m_id
            end
        end
        meilleure_machine !== nothing &&
            log_event(atelier, "FIFO : Employé $emp_id → machine $meilleure_machine")
        return meilleure_machine
    end

    # ---- MODE INTELLIGENT ----
    #
    # Score à maximiser : âge / durée_estimée
    #
    # - Favorise les produits anciens (urgence)
    # - Favorise les tâches courtes
    # - Évite l'omission indéfinie d'une tâche: un produit très vieux finit toujours par avoir
    #   un score suffisant pour être traité, même si sa tâche est longue

    meilleure_machine = nothing
    meilleur_score    = -Inf

    for m_id in emp_ref.qualifications
        machine = atelier.machines[m_id]
        isempty(machine.file_attente) && continue

        idx = index_produit_doyen(machine.file_attente)
        p   = machine.file_attente[idx]

        age           = now(atelier.sim) - p.arrivee_atelier
        a, b          = TEMPS[(p.type, m_id)]
        duree_estimee = (a + b) / 2.0

        # Score: âge pondéré par la durée estimée
        score = age / duree_estimee

        log_event(atelier,
            "Machine $m_id : score=$(round(score, digits=3)) " *
            "(age=$(round(age, digits=3)), durée=$(round(duree_estimee, digits=3)))")

        if score > meilleur_score
            meilleur_score    = score
            meilleure_machine = m_id
        end
    end

    meilleure_machine !== nothing &&
        log_event(atelier,
            "INTELLIGENT  : Employé $emp_id → machine $meilleure_machine " *
            "(score=$(round(meilleur_score, digits=3)))")

    return meilleure_machine
end

function arriver_sur_machine!(atelier::Atelier, p::Produit, m_id::Int, temps::Float64)
    machine = atelier.machines[m_id]
    log_event(atelier, "Produit $(p.id) (type $(p.type)) arrive sur machine $m_id (étape $(p.etape))")
    if machine.en_service === nothing && !machine.occupee
        emp_id = choisir_employe_pour_machine(atelier, m_id)
        if emp_id !== nothing
            machine.occupee = true
            machine.en_service = p
            atelier.employes[emp_id].machine_assignee = m_id
            log_event(atelier, "Machine $m_id libre -> Employé $emp_id assigné, début service immédiat")
            ev = atelier.events_employes[emp_id]
            if state(ev) == SimJulia.idle
                succeed(ev)
            end
        else
            push!(machine.file_attente, p)
            log_event(atelier, "Machine $m_id libre mais aucun employé qualifié disponible -> produit en attente (file=$(length(machine.file_attente)))")
        end
    else
        push!(machine.file_attente, p)
        log_event(atelier, "Machine $m_id occupée -> produit en attente (file=$(length(machine.file_attente)))")
    end
    return nothing
end

# --- PROCESSUS ---

@resumable function processus_employe(sim::Simulation, emp_id::Int, atelier::Atelier,
                                      mode_algo::String, mode_choix::String)
    emp = atelier.employes[emp_id]
    while true
        if !emp.est_occupe
            m_id = choisir_machine(atelier, emp_id, mode_algo, mode_choix)
            if m_id !== nothing
                machine = atelier.machines[m_id]
                machine.occupee = true
                emp.est_occupe = true
                idx_doyen = index_produit_doyen(machine.file_attente)
                p = machine.file_attente[idx_doyen]
                deleteat!(machine.file_attente, idx_doyen)
                machine.en_service = p
                emp.machine_assignee = m_id
                duree = rand(Uniform(TEMPS[(p.type, m_id)]...))
                log_event(atelier, "Employé $emp_id commence service sur machine $m_id pour produit $(p.id) (durée=$(round(duree,digits=3)))")
                @yield timeout(sim, duree)
                if now(sim) >= atelier.fin_transitoire
                    emp.travail_cumule += duree
                end
                machine.en_service = nothing
                machine.occupee = false
                emp.machine_assignee = nothing
                emp.est_occupe = false
                log_event(atelier, "Employé $emp_id termine service sur machine $m_id")
                p.etape += 1
                if p.etape <= length(PARCOURS[p.type])
                    m_suiv = PARCOURS[p.type][p.etape]
                    p.arrivee_machine = now(sim)
                    arriver_sur_machine!(atelier, p, m_suiv, now(sim))
                else
                    t_fin = now(sim)
                    ts = t_fin - p.arrivee_atelier
                    if now(sim) >= atelier.fin_transitoire
                        push!(atelier.temps_sejour, ts)
                    end
                    if atelier.record_history
                        push!(atelier.temps_sejour_history, (t_fin, ts))
                    end
                    log_event(atelier, "Produit $(p.id) termine son parcours (temps séjour=$ts)")
                end
                continue
            end
        end

        if !emp.est_occupe && emp.machine_assignee === nothing
            entrer_salle_attente!(atelier, emp_id, now(sim))
            atelier.events_employes[emp_id] = Event(sim)
            ev = atelier.events_employes[emp_id]
            log_event(atelier, "Employé $emp_id attend un événement (salle d'attente)")
            @yield ev
            m_id = emp.machine_assignee
            emp.machine_assignee = nothing
            if m_id === nothing
                error("Employé $emp_id réveillé sans machine assignée")
            end
            machine = atelier.machines[m_id]
            p = machine.en_service
            duree = rand(Uniform(TEMPS[(p.type, m_id)]...))
            log_event(atelier, "Employé $emp_id réveillé, commence service sur machine $m_id pour produit $(p.id) (durée=$(round(duree,digits=3)))")
            @yield timeout(sim, duree)
            if now(sim) >= atelier.fin_transitoire
                emp.travail_cumule += duree
            end
            machine.en_service = nothing
            machine.occupee = false
            emp.est_occupe = false
            log_event(atelier, "Employé $emp_id termine service sur machine $m_id")
            p.etape += 1
            if p.etape <= length(PARCOURS[p.type])
                m_suiv = PARCOURS[p.type][p.etape]
                p.arrivee_machine = now(sim)
                arriver_sur_machine!(atelier, p, m_suiv, now(sim))
            else
                t_fin = now(sim)
                ts = t_fin - p.arrivee_atelier
                if now(sim) >= atelier.fin_transitoire
                    push!(atelier.temps_sejour, ts)
                end
                if atelier.record_history
                    push!(atelier.temps_sejour_history, (t_fin, ts))
                end
                log_event(atelier, "Produit $(p.id) termine son parcours (temps séjour=$ts)")
            end
            continue
        end
    end
end

@resumable function generateur(sim::Simulation, type::Int, atelier::Atelier)
    while true
        @yield timeout(sim, rand(Exponential(1/λ[type])))
        atelier.compteur_produits += 1
        t_now = now(sim)
        p = Produit(atelier.compteur_produits, type, t_now, t_now, 1)
        log_event(atelier, "Nouveau produit $(p.id) de type $type arrive dans l'atelier")
        premiere_machine = PARCOURS[type][1]
        arriver_sur_machine!(atelier, p, premiere_machine, t_now)
    end
end

@resumable function processus_reinitialisation(sim::Simulation, atelier::Atelier, duree_transient::Float64)
    @yield timeout(sim, duree_transient)
    log_event(atelier, "=== FIN PÉRIODE TRANSITOIRE (t=$duree_transient) - RÉINITIALISATION STATISTIQUES ===")
    for emp in atelier.employes
        emp.travail_cumule = 0.0
    end
end

# --- FONCTION DE SIMULATION ---

function etude_performance(Q, label, mode_algo="FIFO_ATELIER", mode_choix="PLUS_TOT_ATELIER",
                           n_runs=20, duree_transient=10000.0, duree_permanent=1000.0;
                           verbose=false, plot_convergence=false)
    sejours_moyens = Float64[]
    nb_emp = size(Q, 1)
    occupations = [Float64[] for _ in 1:nb_emp]

    for r in 1:n_runs
        record_this_run = plot_convergence && (r == 1)
        verbose && println("\n--- RUN $r ---")
        sim = Simulation()
        atelier = Atelier(sim, Q, verbose, record_this_run)
        atelier.fin_transitoire = duree_transient

        @process processus_reinitialisation(sim, atelier, duree_transient)

        for i in 1:nb_emp
            @process processus_employe(sim, i, atelier, mode_algo, mode_choix)
        end
        for t in 1:4
            @process generateur(sim, t, atelier)
        end

        run(sim, duree_transient + duree_permanent)

        if !isempty(atelier.temps_sejour)
            push!(sejours_moyens, mean(atelier.temps_sejour))
        end
        for i in 1:nb_emp
            push!(occupations[i], atelier.employes[i].travail_cumule / duree_permanent * 100)
        end
        verbose && println("Fin run $r : produits terminés en phase permanente = $(length(atelier.temps_sejour))")

        if record_this_run && !isempty(atelier.temps_sejour_history)
            history = atelier.temps_sejour_history
            sort!(history, by = x -> x[1])
            temps = [t for (t, _) in history]
            sejours = [s for (_, s) in history]
            cum_mean = cumsum(sejours) ./ (1:length(sejours))
            p = plot(temps, cum_mean,
                     label = "Moyenne cumulée",
                     xlabel = "Temps simulé",
                     ylabel = "Temps de séjour moyen",
                     title = "Convergence - Instance $label | Algo: $mode_algo (Run 1)")
            vline!([duree_transient], linestyle=:dash, color=:red, label="Fin transitoire (t=$duree_transient)")
            display(p)
            println("Graphique de convergence affiché pour l'instance $label.")
        end
    end

    m = mean(sejours_moyens)
    ic = 1.96 * std(sejours_moyens) / sqrt(n_runs)
    occ_moy = [mean(occupations[i]) for i in 1:nb_emp]

    println("-"^60)
    @printf("Instance %s | Algo: %-15s | Choix: %-15s\n", label, mode_algo, mode_choix)
    @printf(" > Temps de séjour moyen : %.4f ± %.4f\n", m, ic)
    @printf(" > Occupation employés   : %s\n", join([@sprintf("%.1f%%", x) for x in occ_moy], ", "))
end

# --- EXÉCUTION ---
Q1 = [1 1 0 0 0 0 0 0; 0 0 1 1 0 0 0 0; 0 0 0 0 1 1 0 0; 0 0 0 0 0 0 1 1]
Q2 = [1 0 1 0 0 1 0 0; 0 1 0 0 1 0 1 1; 0 1 0 1 1 0 0 1; 1 0 1 1 0 1 1 0]
Q3 = [1 1 0 0 0 0 0 0; 0 0 1 0 0 0 0 0; 0 0 0 1 0 1 0 0; 0 0 0 0 1 0 0 1; 0 0 1 0 0 1 0 0; 1 0 0 0 0 0 1 0]
Q4 = [1 1 1 0 0 0 0 0; 0 0 0 1 1 1 0 0; 1 0 1 0 0 1 1 1; 0 0 1 0 1 0 1 1; 0 1 0 0 0 1 1 0; 1 0 0 1 0 0 1 1]

Random.seed!(123)
println("="^60)
println("ALGO FIFO (Q4)")
println("="^60)
etude_performance(Q1, "I1", "FIFO_ATELIER", "PLUS_TOT_ATELIER", plot_convergence=true)
etude_performance(Q2, "I2", "FIFO_ATELIER", "PLUS_TOT_ATELIER", plot_convergence=true)
etude_performance(Q3, "I3", "FIFO_ATELIER", "PLUS_TOT_ATELIER", plot_convergence=true)
etude_performance(Q4, "I4", "FIFO_ATELIER", "PLUS_TOT_ATELIER", plot_convergence=true)

println()
println("="^60)
println("ALGO INTELLIGENT (Q6)")
println("="^60)
etude_performance(Q1, "I1", "INTELLIGENT", "PLUS_TOT_ATELIER")
etude_performance(Q2, "I2", "INTELLIGENT", "PLUS_TOT_ATELIER")
etude_performance(Q3, "I3", "INTELLIGENT", "PLUS_TOT_ATELIER")
etude_performance(Q4, "I4", "INTELLIGENT", "PLUS_TOT_ATELIER")

**Résultats:**
Les simulations ont été conduites dans les mêmes conditions que pour l'algorithme FIFO (régime permanent estimé, intervalles de confiance à 95 %). Voici les résultats obtenus :

| Instance | FIFO — Temps de séjour | Intelligent — Temps de séjour |   Δ   |
|:--------:|:----------------------:|:----------------------:|:-----:|
|    I1    |    5,3696 ± 0,3500     |    5,1152 ± 0,1779     | −4,7 % |
|    I2    |    3,4692 ± 0,0844     |    3,4279 ± 0,0702     | −1,2 % |
|    I3    |    3,7962 ± 0,1233     |    3,6694 ± 0,0904     | −3,3 % |
|    I4    |    2,4410 ± 0,0127     |    2,4441 ± 0,0136     | +0,1 % |


L'algorithme intelligent améliore le temps de séjour moyen sur trois des quatre instances, avec des gains modestes mais cohérents sur I1, I2 et I3. Sur I4, la différence est négligeable et les intervalles de confiance se chevauchent largement : les deux algorithmes sont statistiquement équivalents sur cette instance.

**Sur I1**, le gain est le plus visible en valeur absolue (−0,25 unité de temps), et l'intervalle de confiance du nouvel algorithme est nettement plus étroit que celui de FIFO (±0,18 contre ±0,35). Il produit donc non seulement de meilleures performances en moyenne, mais aussi des résultats plus stables d'une simulation à l'autre. Cela s'explique par le fait que ce-dernier évite les situations que FIFO peut laisser apparaître : aucun produit n'attend indéfiniment, ce qui lisse les temps de séjour extrêmes.

**Sur I3**, le gain de −3,3 % est statistiquement crédible car les intervalles de confiance ne se chevauchent pas. I3 est une instance avec 6 employés aux qualifications relativement restreintes, ce qui crée des déséquilibres de charge marqués (occupations allant de 27,7 % à 76,6 %). Dans ce contexte, notre algorithme tire mieux parti des employés disponibles en priorisant les produits anciens sur les machines peu couvertes.

**Sur I4**, les qualifications sont larges et les taux d'occupation sont bien équilibrés (entre 43,8 % et 54,5 %). Le système est suffisamment flexible pour que le choix de routage ait peu d'impact : FIFO et l'algorithme intelligent convergent vers le même comportement car il y a presque toujours un employé disponible rapidement, quelle que soit la machine choisie.

Les taux d'occupation des employés restent quasi identiques entre les deux algorithmes sur toutes les instances. Cela confirme que la politique de routage n'affecte pas la charge globale du système — seul l'ordre dans lequel les tâches sont traitées change, pas leur quantité totale.

## Question 7 : Inégalités entre les employés

Il y a plusieurs raisons strucuturelles qui expliquent la répartition inéquilibrée des différentes tâches entre les employés. 

**Premièrement :** Les matrices $Q$ montrent que les employés ne couvrent pas le même nombre de machines, ni les mêmes machines. Par exemple pour l'instance $I1$, l'employé 1 couvre $M1, M2$, le 2 couvre $M3, M4$, le 3 couvre $M5, M6$ et le 4 couvre $M7, M8$. Ainsi, chaque employé est spécifiquement en charge de certaines machines. Sa charge est donc entièrement déterminée par les flux de celles-ci sans partage possible.

**Deuxièmement :** Toutes les machines ne reçoivent pas le même flux de produits. En croisant les tables d'intensités d'arrivée et les parcours, on remarque que $M1$ et $M2$ sont visitées par $T1, T2, T3, T4$ avec des temps de traitement différents. Egalement, d'autres machines ne sont visitées que par un ou deux types. C'est le cas de $M7, M8$.
Les employés qualifiés sur des machines très fréquentées travaillent donc naturellement plus que ceux affectés à des machines peu sollicitées. Cela se fait indépendamment de l'algorithme utilisé.

**Question 8:**

**Question 9:**

**Question 10:**